In [0]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import DataFrame
import pyspark.sql.functions as F
import pyspark.sql.functions as f
import sys
from pyspark.sql.window import Window
import logging
import os
from abc import ABC, abstractmethod


In [0]:
now = datetime.now(ZoneInfo("Asia/Kolkata"))
currweek = now.strftime('%Y%W')
currdate = now.strftime("%Y%m%d")
currdatetime = now.strftime('%Y-%m-%d %H:%M:%S')

currweek,currdate

In [0]:
# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

VOLUME_BASE = "/Volumes/workspace/m5"

LOG_DIR = f"{VOLUME_BASE}/logs"

# Create logs folder if it doesn't exist
dbutils.fs.mkdirs(LOG_DIR)


# ============================================================
# 2. INDIA TIME
# ============================================================

def get_ist_time():
    return datetime.now(
        ZoneInfo("Asia/Kolkata")
    )


# ============================================================
# 3. LOG FILE
# ============================================================

currdate = get_ist_time().strftime("%Y%m%d")

LOG_FILE = (
    f"{LOG_DIR}/"
    f"m5_forecasting_{currdate}.log"
)


# ============================================================
# 4. CUSTOM DATABRICKS VOLUME LOG HANDLER
# ============================================================

class DatabricksVolumeHandler(logging.Handler):

    def emit(self, record):

        try:
            # Format log message
            log_message = self.format(record) + "\n"

            # Read existing log file
            try:
                existing_logs = dbutils.fs.head(
                    LOG_FILE,
                    10 * 1024 * 1024
                )
            except Exception:
                existing_logs = ""

            # Append new log
            updated_logs = existing_logs + log_message

            # Write back to Volume
            dbutils.fs.put(
                LOG_FILE,
                updated_logs,
                overwrite=True
            )

        except Exception:
            self.handleError(record)


# ============================================================
# 5. LOGGER CONFIGURATION
# ============================================================

logger = logging.getLogger("m5_forecasting")

logger.setLevel(logging.INFO)


# Prevent duplicate handlers if notebook cell is executed again
if logger.handlers:
    logger.handlers.clear()


formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)


# ============================================================
# 6. CONSOLE LOGGER
# ============================================================

console_handler = logging.StreamHandler()

console_handler.setFormatter(formatter)

logger.addHandler(console_handler)


# ============================================================
# 7. VOLUME LOGGER
# ============================================================

volume_handler = DatabricksVolumeHandler()

volume_handler.setFormatter(formatter)

logger.addHandler(volume_handler)


logger.propagate = False

In [0]:
%run ./utils

In [0]:
raw_paths = {
   'sales' :  "/Volumes/workspace/m5/raw/sales.csv",
   'calendar' : "/Volumes/workspace/m5/raw/calendar.csv",
   "price" : "/Volumes/workspace/m5/raw/sell_prices.csv",
}

processed_paths = {
    'processed_sales' : '/Volumes/workspace/m5/processed/sales',
    'processed_price' : '/Volumes/workspace/m5/processed/price',
    'processed_events' : '/Volumes/workspace/m5/processed/events',
    'joined_data' : '/Volumes/workspace/m5/processed/joined_data',
    'FE' : '/Volumes/workspace/m5/processed/FE',
}

In [0]:
filter_map = {
    'key_col' : 'id',
    'yearweek_col' : 'yearweek',
    'sales_col' : 'sales',
    'sales_id_cols' : ['id','item_id','dept_id','cat_id','store_id','state_id'],
}

In [0]:
m5_config = {
    'id_col' : 'id',
    'yearweek_col' : 'yearweek',
    'week_col' : 'week',
    'product_col' : 'item_id',
    'department_col' : 'dept_id',
    'category_col' : 'cat_id',
    'customer_col' : 'store_id',
    'state_col' : 'state_id',
    'event_flag' : 'event_flag',
    'y' : 'sales',
    'price_col' : 'sell_price',
    'lags_features' : [1,4,8,13,26,52],
    'rolling_features' : {
        'mean' : [4,8,13,26],
        'std' : [4,8],
        'min' : [4,8],
        'max' : [4,8]
    },
    'event_lead_lags_features' : ['f_event_christmas','f_event_newyear','f_event_thanksgiving','f_event_laborday'],


}

In [0]:
def get_raw_input_data(raw_paths):
    """reads the raw datasets and returns them as dictionary of datasets"""
    raw_sales = read_data(raw_paths['sales'],format='csv')
    raw_calendar = read_data(raw_paths['calendar'],format='csv')
    raw_price = read_data(raw_paths['price'],format='csv')
    return {'raw_sales':raw_sales,
            'calendar':raw_calendar,
            'raw_price':raw_price}
    

